In [ ]:
import pandas as pd
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Nạp dữ liệu thành phẩm từ File 1
df_train = pd.read_csv('data/processed/train_final.csv').dropna()
df_val = pd.read_csv('data/processed/val_final.csv').dropna()
df_test_yt = pd.read_csv('data/processed/test_youtube.csv').dropna()

# 2. Khởi tạo TF-IDF (Lấy 5000 từ quan trọng)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# 3. HỌC TỪ ĐIỂN (Chỉ Fit trên tập Train để tránh rò rỉ dữ liệu)
X_train = vectorizer.fit_transform(df_train['text_cleaned'])

# 4. ÁP DỤNG cho các tập còn lại
X_val = vectorizer.transform(df_val['text_cleaned'])
X_test_yt = vectorizer.transform(df_test_yt['text_cleaned'])

y_train = df_train['label']
y_val = df_val['label']
y_test_yt = df_test_yt['label']

# 5. Lưu lại Vectorizer để dùng cho các File 3, 4 sau này
if not os.path.exists('saved_models'): os.makedirs('saved_models')
joblib.dump(vectorizer, 'saved_models/tfidf_vectorizer.pkl')

print(f"✅ Đã tạo ma trận số. Số lượng từ vựng AI học được: {len(vectorizer.get_feature_names_out())}")

✅ Đã tạo ma trận số. Số lượng từ vựng AI học được: 5000


In [6]:
def calculate_oov(vectorizer, text_series):
    vocab = set(vectorizer.get_feature_names_out())
    total_words = 0
    oov_count = 0
    for text in text_series:
        words = str(text).split()
        for w in words:
            total_words += 1
            if w not in vocab:
                oov_count += 1
    return (oov_count / total_words) * 100 if total_words > 0 else 0

print(f" Tỷ lệ OOV trên YouTube: {calculate_oov(vectorizer, df_test_yt['text_cleaned']):.2f}%")

 Tỷ lệ OOV trên YouTube: 33.10%


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train)
val_acc_lr = accuracy_score(y_val, lr.predict(X_val))
yt_acc_lr = accuracy_score(y_test_yt, lr.predict(X_test_yt))

# 2. Random Forest (Mặc định)
rf = RandomForestClassifier(random_state=42, class_weight='balanced').fit(X_train, y_train)
val_acc_rf = accuracy_score(y_val, rf.predict(X_val))
yt_acc_rf = accuracy_score(y_test_yt, rf.predict(X_test_yt))

# 3. XGBoost (Mặc định)
xgb = XGBClassifier(random_state=42).fit(X_train, y_train)
val_acc_xgb = accuracy_score(y_val, xgb.predict(X_val))
yt_acc_xgb = accuracy_score(y_test_yt, xgb.predict(X_test_yt))

print(f"{'Thuật toán':<25} | {'Acc Val (Chuẩn)':<18} | {'Acc YouTube (Thực)':<18} | {'Độ sụt giảm'}")
print("-" * 85)
print(f"{'Logistic Regression':<25} | {val_acc_lr*100:15.2f}% | {yt_acc_lr*100:15.2f}% | {(val_acc_lr-yt_acc_lr)*100:.2f}%")
print(f"{'Random Forest':<25} | {val_acc_rf*100:15.2f}% | {yt_acc_rf*100:15.2f}% | {(val_acc_rf-yt_acc_rf)*100:.2f}%")
print(f"{'XGBoost':<25} | {val_acc_xgb*100:15.2f}% | {yt_acc_xgb*100:15.2f}% | {(val_acc_xgb-yt_acc_xgb)*100:.2f}%")

Thuật toán                | Acc Val (Chuẩn)    | Acc YouTube (Thực) | Độ sụt giảm
-------------------------------------------------------------------------------------
Logistic Regression       |           83.46% |           57.47% | 25.99%
Random Forest             |           88.47% |           64.47% | 24.00%
XGBoost                   |           88.82% |           66.88% | 21.95%
